# Guardar un modelo entrenado con `joblib` y `pickle`

**Módulo 06 — Fundamentos de redes neuronales.** Recurso de las clases 23-24, sobre persistencia de modelos.

## Por qué persistir un modelo

Entrenar un `MLPClassifier` (o cualquier modelo) tiene un costo en tiempo y en cómputo. Si cada vez que se necesita predecir hay que volver a entrenar desde cero, ese costo se paga una y otra vez. La **persistencia de modelos** consiste en guardar el objeto ya entrenado (con todos sus pesos y parámetros) en un archivo, para poder reutilizarlo después sin reentrenar. Esto habilita varios escenarios de producción:

- **Servir el modelo en tiempo real**: cargar el archivo una vez y responder predicciones sobre la marcha, sin reentrenar en cada request.
- **Desplegar varias instancias**: copiar el mismo archivo del modelo a múltiples servidores o contenedores, garantizando que todos respondan exactamente igual.
- **Versionar modelos**: guardar cada entrenamiento con su propio archivo permite comparar versiones y, si una nueva versión anda peor, volver atrás (rollback) a una anterior.

En este notebook se entrena un `MLPClassifier` chico sobre Iris y se lo guarda de dos formas distintas: con `joblib` y con `pickle`.

In [19]:
from sklearn.datasets import load_iris
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
import pandas as pd
import joblib
import pickle

## Imports

Se usa `load_iris` para el dataset, `MLPClassifier` como modelo, `train_test_split` para separar entrenamiento y prueba, `pandas` para manipular los datos, y `joblib` / `pickle` como las dos librerías de persistencia que se van a comparar.

In [10]:
#Cargar el dataset
iris = load_iris()

## Carga del dataset

Se carga el dataset Iris con `load_iris()` y se arma un `DataFrame` de pandas con sus atributos y la columna `target` (la especie).

In [11]:
#Leer el dataset
data_df = pd.DataFrame(iris.data, columns=iris.feature_names)
data_df["target"] = iris.target

## Selección de atributos

Para mantener el ejemplo simple (y el modelo resultante liviano), se trabaja solo con dos atributos: `sepal length (cm)` y `petal length (cm)`, además del `target`.

In [12]:
#Seleccionar columnas
columns = ["sepal length (cm)", "petal length (cm)", "target"]
data_df = data_df[columns]

## Separación de `X` e `y`

Se extraen los valores del DataFrame: `X` con los dos atributos seleccionados e `y` con la especie. El **orden** de las columnas en `X` (primero sepal length, después petal length) es importante: es el orden con el que se va a entrenar el modelo, y es el mismo orden que va a haber que respetar más adelante al pedirle predicciones.

In [13]:
#Obtenemos solo los valores de nuestro DataFrame
X = data_df[["sepal length (cm)", "petal length (cm)"]].values
y = data_df["target"].values

## Split de entrenamiento y prueba

Se separa un 20% de los datos para prueba (`test_size=0.2`), con una semilla fija (`random_state=42`) para que el split sea reproducible.

In [14]:
#Dividir datos para entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Entrenamiento del modelo

Se entrena un `MLPClassifier` con dos capas ocultas (100 y 150 neuronas), activación `relu` y un máximo de 300 iteraciones. Una vez que `mlp.fit()` termina, el objeto `mlp` ya tiene todos los pesos ajustados: es este objeto el que hay que persistir para no tener que repetir el entrenamiento.

In [15]:
#Entrenar el modelo
mlp = MLPClassifier(hidden_layer_sizes=(100,150,), max_iter=300, activation='relu', random_state=42)
mlp.fit(X_train, y_train)

c:\Users\MSI\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(hidden_layer_sizes=(100, 150), max_iter=300, random_state=42)

## Guardar el modelo con `joblib`

`joblib.dump(objeto, archivo)` serializa el objeto Python (el modelo entrenado) a un archivo en disco. `joblib` es la opción recomendada por scikit-learn para persistir modelos, porque maneja de forma eficiente los arrays de NumPy que suelen representar los pesos.

Acá se guardan **dos** versiones del mismo modelo:

- `mi_modelo.joblib`: sin comprimir.
- `mi_modelo_comprimido.joblib`: con `compress=5`, un parámetro que va de `0` (sin compresión) a `9` (compresión máxima). A mayor compresión, más chico el archivo pero más tiempo toma guardarlo y cargarlo — es un trade-off entre espacio en disco y velocidad.

**Dato medido en este notebook**: el archivo sin comprimir pesa `392.276` bytes y el comprimido `375.378` bytes, apenas un 4% menos. Con un modelo tan chico como este la compresión casi no aporta; donde realmente rinde es en modelos grandes (muchas capas, muchos parámetros), donde la diferencia de tamaño sí es significativa.

Nota: la slide de la clase usa `compress=3`; acá se usa `compress=5` a modo de ejemplo. Ambos son valores válidos dentro del rango 0-9.

In [22]:
#Guardar modelo con joblib
joblib.dump(mlp, 'mi_modelo.joblib')

joblib.dump(mlp, 'mi_modelo_comprimido.joblib', compress=5)

['mi_modelo_comprimido.joblib']

## Guardar el modelo con `pickle`

`pickle` es el módulo estándar de Python para serializar objetos (no es específico de machine learning, sirve para cualquier objeto Python). Para guardar el modelo, se abre un archivo en modo binario de escritura (`'wb'`) y se llama a `pickle.dump(objeto, archivo)`.

A diferencia de `joblib.dump`, acá el archivo se abre explícitamente con `open(...)` antes de pasarlo a `pickle.dump`.

In [20]:
#Guardar modelo con pickle
with open('mi_modelo.pkl', 'wb') as archivo:
    pickle.dump(mlp, archivo)

## Valores de prueba

Se imprimen `X_test` e `y_test` para tener a mano ejemplos reales del dataset con los que más adelante se va a probar el modelo ya cargado, en los notebooks `cargar_modelo_joblib.ipynb` y `cargar_modelo_pickle.ipynb`.

In [18]:
print(X_test)
print(y_test)

[[6.1 4.7]
 [5.7 1.7]
 [7.7 6.9]
 [6.  4.5]
 [6.8 4.8]
 [5.4 1.5]
 [5.6 3.6]
 [6.9 5.1]
 [6.2 4.5]
 [5.8 3.9]
 [6.5 5.1]
 [4.8 1.4]
 [5.5 1.3]
 [4.9 1.5]
 [5.1 1.5]
 [6.3 4.7]
 [6.5 5.8]
 [5.6 3.9]
 [5.7 4.5]
 [6.4 5.6]
 [4.7 1.6]
 [6.1 4.9]
 [5.  1.6]
 [6.4 5.6]
 [7.9 6.4]
 [6.7 5.2]
 [6.7 5.8]
 [6.8 5.9]
 [4.8 1.4]
 [4.8 1.6]]
[1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]


## Advertencia: los archivos generados no están versionados

Los archivos `.joblib` y `.pkl` que genera este notebook (`mi_modelo.joblib`, `mi_modelo_comprimido.joblib`, `mi_modelo.pkl`) **no se suben al repositorio**: como todo artefacto binario, están excluidos por `.gitignore`.

Por eso, para que `cargar_modelo_joblib.ipynb` y `cargar_modelo_pickle.ipynb` funcionen, hace falta **ejecutar este notebook primero** y generar los archivos localmente.